# YouTube Performance and Community Health Analysis

This notebook answers the beginner, intermediate, and advanced project questions in a simpler step-by-step format.

Note:
- The beginner and intermediate questions are answered directly.
- The advanced moderation monitor is a risk proxy, not a true sentiment spiral detector, because the dataset has no comment timestamps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
colors = {
    "blue": "#4C78A8",
    "orange": "#F58518",
    "green": "#54A24B",
    "red": "#E45756",
}

## Load And Inspect The Data

First, read the two source files and get a quick sense of their shape.

In [ ]:
videos_raw = pd.read_csv("videos_stats.csv")
comments_raw = pd.read_csv("comments.csv")

In [ ]:
print("Videos shape:", videos_raw.shape)
print("Comments shape:", comments_raw.shape)

display(pd.DataFrame({
    "videos_columns": pd.Series(videos_raw.columns),
    "comments_columns": pd.Series(comments_raw.columns),
}))

In [ ]:
# Preview the video-level file
display(videos_raw.head())

In [ ]:
# Preview the comment-level file
display(comments_raw.head())

In [ ]:
# Check missing values in both source tables
display(videos_raw.isna().sum().rename("missing_values").to_frame())
display(comments_raw.isna().sum().rename("missing_values").to_frame())

In [ ]:
# Check duplicate videos and comment sample sizes
duplicate_videos = videos_raw["Video ID"].duplicated().sum()
comment_samples = comments_raw["Video ID"].value_counts()

print("Duplicate video rows:", duplicate_videos)
print("Videos with comment samples:", comment_samples.size)
print("Average sampled comments per video:", round(comment_samples.mean(), 2))

In [ ]:
# Visualize how many sampled comments each video has
plt.figure(figsize=(10, 5))
sns.histplot(comment_samples, bins=20, color=colors["blue"])
plt.title("Sampled Comments Per Video")
plt.xlabel("Number of sampled comments")
plt.ylabel("Number of videos")
plt.tight_layout()
plt.show()

## Clean The Source Tables

Here I standardize column names, fix data types, and collapse duplicate video rows.

In [ ]:
# Rename the video columns to simpler snake_case names
videos = videos_raw.rename(columns={
    "Title": "title",
    "Video ID": "video_id",
    "Published At": "published_at",
    "Keyword": "keyword",
    "Likes": "likes",
    "Comments": "comments",
    "Views": "views",
})

In [ ]:
# Keep only the columns used in the analysis
videos = videos[["title", "video_id", "published_at", "keyword", "likes", "comments", "views"]].copy()

In [ ]:
# Convert engagement columns to numbers
numeric_video_cols = ["likes", "comments", "views"]
videos[numeric_video_cols] = videos[numeric_video_cols].apply(pd.to_numeric, errors="coerce")
videos[numeric_video_cols] = videos[numeric_video_cols].mask(videos[numeric_video_cols] < 0)

In [ ]:
# Parse the publish date
videos["published_at"] = pd.to_datetime(videos["published_at"], errors="coerce")

In [ ]:
# Keep one row per video for the core performance stats
video_base = videos.groupby("video_id", as_index=False).agg(
    title=("title", "first"),
    published_at=("published_at", "min"),
    views=("views", "max"),
    likes=("likes", "max"),
    comments=("comments", "max"),
)

In [ ]:
# Keep all distinct keywords attached to each video
keyword_lists = videos.groupby("video_id")["keyword"].apply(
    lambda s: sorted({str(x).strip() for x in s.dropna() if str(x).strip()})
).reset_index(name="keywords")

In [ ]:
# Merge the base stats and keyword lists
videos_clean = video_base.merge(keyword_lists, on="video_id", how="left")
videos_clean["keywords_joined"] = videos_clean["keywords"].apply(lambda x: " | ".join(x) if isinstance(x, list) else "")
videos_clean["keyword_count"] = videos_clean["keywords"].apply(lambda x: len(x) if isinstance(x, list) else 0)

In [ ]:
# Preview the cleaned video table
display(videos_clean.head())

In [ ]:
# Rename comment columns to snake_case
comments = comments_raw.rename(columns={
    "Video ID": "video_id",
    "Comment": "comment",
    "Likes": "likes",
    "Sentiment": "sentiment",
})

In [ ]:
# Keep only the columns used in the comment analysis
comments = comments[["video_id", "comment", "likes", "sentiment"]].copy()

In [ ]:
# Convert comment likes and sentiment labels to numbers
comments[["likes", "sentiment"]] = comments[["likes", "sentiment"]].apply(pd.to_numeric, errors="coerce")

In [ ]:
# Count how many negative, neutral, and positive comments each video has
sentiment_counts = comments.groupby(["video_id", "sentiment"]).size().unstack(fill_value=0)
sentiment_counts = sentiment_counts.rename(columns={0: "negative_comments", 1: "neutral_comments", 2: "positive_comments"})
sentiment_counts = sentiment_counts.rename(columns={0.0: "negative_comments", 1.0: "neutral_comments", 2.0: "positive_comments"})
sentiment_counts = sentiment_counts.reset_index()

In [ ]:
# Add missing sentiment columns if one class is absent
for col in ["negative_comments", "neutral_comments", "positive_comments"]:
    if col not in sentiment_counts.columns:
        sentiment_counts[col] = 0

sentiment_counts = sentiment_counts[["video_id", "negative_comments", "neutral_comments", "positive_comments"]]

In [ ]:
# Compute simple averages for each video's sampled comments
comment_means = comments.groupby("video_id", as_index=False).agg(
    sampled_comments=("comment", "size"),
    avg_sentiment=("sentiment", "mean"),
    avg_comment_likes=("likes", "mean"),
)

In [ ]:
# Combine the sentiment counts and comment averages
comment_summary = sentiment_counts.merge(comment_means, on="video_id", how="outer")
display(comment_summary.head())

In [ ]:
# Merge the video table with the comment table
video_metrics = videos_clean.merge(comment_summary, on="video_id", how="left")

In [ ]:
# Fill missing comment metrics with zeros
comment_cols = [
    "negative_comments",
    "neutral_comments",
    "positive_comments",
    "sampled_comments",
    "avg_sentiment",
    "avg_comment_likes",
]
video_metrics[comment_cols] = video_metrics[comment_cols].fillna(0)

In [ ]:
# Create the main beginner metrics
video_metrics["engagement_score"] = video_metrics["views"] + video_metrics["likes"] + video_metrics["comments"]
video_metrics["like_to_view_ratio"] = video_metrics["likes"] / video_metrics["views"].replace(0, np.nan)
video_metrics["comment_to_view_ratio"] = video_metrics["comments"] / video_metrics["views"].replace(0, np.nan)

In [ ]:
# Convert comment counts into simple sentiment shares
video_metrics["negative_ratio"] = video_metrics["negative_comments"] / video_metrics["sampled_comments"].replace(0, np.nan)
video_metrics["neutral_ratio"] = video_metrics["neutral_comments"] / video_metrics["sampled_comments"].replace(0, np.nan)
video_metrics["positive_ratio"] = video_metrics["positive_comments"] / video_metrics["sampled_comments"].replace(0, np.nan)

In [ ]:
# Measure how balanced the positive and negative comments are
polar_ratio = video_metrics["positive_ratio"].fillna(0) + video_metrics["negative_ratio"].fillna(0)
gap = (video_metrics["positive_ratio"].fillna(0) - video_metrics["negative_ratio"].fillna(0)).abs()
video_metrics["controversy_score"] = np.where(polar_ratio > 0, 1 - (gap / polar_ratio), 0)
video_metrics["sample_confidence"] = (video_metrics["sampled_comments"] / 10).clip(upper=1)

In [ ]:
# Create time-based features for the intermediate analysis
today = pd.Timestamp.today().normalize()
video_metrics["days_live"] = (today - video_metrics["published_at"]).dt.days.clip(lower=1)
video_metrics["publish_year"] = video_metrics["published_at"].dt.year.astype("Int64")
video_metrics["publish_month"] = video_metrics["published_at"].dt.to_period("M").astype(str)
video_metrics["engagement_per_day"] = video_metrics["engagement_score"] / video_metrics["days_live"]

In [ ]:
# Preview the final analysis table
display(video_metrics[[
    "title", "keywords_joined", "engagement_score", "avg_sentiment", "negative_ratio", "controversy_score"
]].head())

## Beginner Analysis

Goal:
- Rank videos with an engagement score.
- Find strong keywords.
- Compare like-to-view ratio and average sentiment.

In [ ]:
# Rank videos by the required engagement score
beginner_top_videos = video_metrics[[
    "title", "keywords_joined", "views", "likes", "comments",
    "engagement_score", "like_to_view_ratio", "avg_sentiment"
]].sort_values("engagement_score", ascending=False).head(10)

display(beginner_top_videos)

In [ ]:
# Visualize the top videos by engagement score
chart_data = beginner_top_videos.sort_values("engagement_score")
plt.figure(figsize=(11, 6))
plt.barh(chart_data["title"], chart_data["engagement_score"], color=colors["blue"])
plt.title("Top 10 Videos By Engagement Score")
plt.xlabel("Engagement score")
plt.ylabel("Video title")
plt.tight_layout()
plt.show()

In [ ]:
# Split videos back out by keyword for category-level analysis
keyword_metrics = video_metrics.explode("keywords").rename(columns={"keywords": "keyword"})
keyword_metrics = keyword_metrics.dropna(subset=["keyword"])

In [ ]:
# Summarize video success by keyword
keyword_summary = keyword_metrics.groupby("keyword", as_index=False).agg(
    videos=("video_id", "nunique"),
    avg_engagement_score=("engagement_score", "mean"),
    median_engagement_score=("engagement_score", "median"),
    avg_like_to_view_ratio=("like_to_view_ratio", "mean"),
    avg_sentiment=("avg_sentiment", "mean"),
    avg_negative_ratio=("negative_ratio", "mean"),
)

In [ ]:
# Turn the keyword metrics into one simple ranking score
keyword_summary["keyword_success_score"] = (
    0.45 * keyword_summary["avg_engagement_score"].rank(pct=True)
    + 0.25 * keyword_summary["avg_like_to_view_ratio"].rank(pct=True)
    + 0.20 * keyword_summary["avg_sentiment"].rank(pct=True)
    + 0.10 * keyword_summary["videos"].rank(pct=True)
)
keyword_summary = keyword_summary.sort_values("keyword_success_score", ascending=False)

In [ ]:
# Show the strongest keyword groups
display(keyword_summary.head(15))

In [ ]:
# Visualize the strongest keywords
top_keywords = keyword_summary.head(12).sort_values("keyword_success_score")
plt.figure(figsize=(10, 6))
plt.barh(top_keywords["keyword"], top_keywords["keyword_success_score"], color=colors["orange"])
plt.title("Top Keywords By Success Score")
plt.xlabel("Keyword success score")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()

In [ ]:
# Use the strongest keywords in a simple quality map
keyword_scatter = keyword_summary.head(15).copy()
keyword_scatter["marker_size"] = np.sqrt(keyword_scatter["median_engagement_score"].clip(lower=1)) / 8

In [ ]:
# Start the keyword quality plot
plt.figure(figsize=(11, 7))
scatter = plt.scatter(
    keyword_scatter["avg_like_to_view_ratio"],
    keyword_scatter["avg_sentiment"],
    s=keyword_scatter["marker_size"],
    c=keyword_scatter["keyword_success_score"],
    cmap="viridis",
    alpha=0.8,
)

In [ ]:
# Add labels and finish the keyword quality plot
for _, row in keyword_scatter.iterrows():
    plt.text(row["avg_like_to_view_ratio"], row["avg_sentiment"], row["keyword"], fontsize=9)

plt.title("Keyword Quality Map")
plt.xlabel("Average like-to-view ratio")
plt.ylabel("Average sentiment")
plt.colorbar(scatter, label="Keyword success score")
plt.tight_layout()
plt.show()

## Intermediate Analysis

Goal:
- Evaluate content more fairly over time.
- Use percentiles and cohorts.
- Check how sentiment relates to performance.

In [ ]:
# Build percentiles for the main quality signals
video_metrics["like_to_view_percentile"] = video_metrics["like_to_view_ratio"].rank(pct=True)
video_metrics["sentiment_percentile"] = video_metrics["avg_sentiment"].rank(pct=True)
video_metrics["negative_sentiment_percentile"] = video_metrics["negative_ratio"].rank(pct=True)
video_metrics["controversy_percentile"] = video_metrics["controversy_score"].rank(pct=True)

In [ ]:
# Add percentiles for scale-related metrics used later in moderation
video_metrics["comment_volume_percentile"] = video_metrics["comments"].rank(pct=True)
video_metrics["reach_percentile"] = video_metrics["views"].rank(pct=True)

In [ ]:
# Compare each video against others from the same publish year
video_metrics["cohort_engagement_percentile"] = video_metrics.groupby("publish_year")["engagement_per_day"].rank(pct=True)

In [ ]:
# Create one performance score from cohort performance, like ratio, and sentiment
video_metrics["performance_framework_score"] = 100 * (
    0.50 * video_metrics["cohort_engagement_percentile"].fillna(0)
    + 0.30 * video_metrics["like_to_view_percentile"].fillna(0)
    + 0.20 * video_metrics["sentiment_percentile"].fillna(0)
)

In [ ]:
# Turn the framework score into readable performance bands
video_metrics["performance_band"] = np.select(
    [
        video_metrics["performance_framework_score"] >= 90,
        video_metrics["performance_framework_score"] >= 75,
        video_metrics["performance_framework_score"] >= 50,
        video_metrics["performance_framework_score"] >= 25,
    ],
    ["elite", "strong", "steady", "developing"],
    default="weak",
)

In [ ]:
# Preview the performance framework output
display(video_metrics[[
    "title", "publish_year", "engagement_per_day", "cohort_engagement_percentile",
    "performance_framework_score", "performance_band"
]].head(15))

In [ ]:
# Summarize content performance by publish year
year_summary = video_metrics.groupby("publish_year", as_index=False).agg(
    videos=("video_id", "nunique"),
    avg_engagement_per_day=("engagement_per_day", "mean"),
    avg_performance_score=("performance_framework_score", "mean"),
    avg_sentiment=("avg_sentiment", "mean"),
)

In [ ]:
# View the yearly cohort summary
display(year_summary.sort_values("publish_year"))

In [ ]:
# Plot engagement per day by cohort year
plot_years = year_summary.dropna(subset=["publish_year"]).sort_values("publish_year")
plt.figure(figsize=(10, 5))
plt.plot(plot_years["publish_year"], plot_years["avg_engagement_per_day"], marker="o", color=colors["green"])
plt.title("Average Engagement Per Day By Publish Year")
plt.xlabel("Publish year")
plt.ylabel("Average engagement per day")
plt.tight_layout()
plt.show()

In [ ]:
# Use a safe correlation helper so very small cohorts do not raise warnings
def safe_corr(x, y):
    pair = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(pair) < 2 or pair["x"].nunique() < 2 or pair["y"].nunique() < 2:
        return np.nan
    return pair["x"].corr(pair["y"])

In [ ]:
# Check how sentiment metrics relate to performance overall
overall_correlations = pd.Series({
    "avg_sentiment_vs_engagement_per_day": safe_corr(video_metrics["avg_sentiment"], video_metrics["engagement_per_day"]),
    "negative_ratio_vs_engagement_per_day": safe_corr(video_metrics["negative_ratio"], video_metrics["engagement_per_day"]),
    "controversy_vs_engagement_per_day": safe_corr(video_metrics["controversy_score"], video_metrics["engagement_per_day"]),
}).rename_axis("metric").reset_index(name="correlation")

In [ ]:
# View the overall correlation table
display(overall_correlations)

In [ ]:
# Check whether the same pattern holds inside each year cohort
yearly_correlations = video_metrics.dropna(subset=["publish_year"]).groupby("publish_year")[[
    "avg_sentiment", "negative_ratio", "engagement_per_day"
]].apply(
    lambda df: pd.Series({
        "sentiment_vs_engagement_per_day": safe_corr(df["avg_sentiment"], df["engagement_per_day"]),
        "negative_ratio_vs_engagement_per_day": safe_corr(df["negative_ratio"], df["engagement_per_day"]),
    })
).reset_index()

In [ ]:
# View the cohort-level correlation table
display(yearly_correlations)

In [ ]:
# Highlight strong performers from the most recent publish year
latest_year = int(video_metrics["publish_year"].dropna().max())
recent_standouts = video_metrics.loc[video_metrics["publish_year"] == latest_year, [
    "title", "keywords_joined", "engagement_per_day", "cohort_engagement_percentile",
    "performance_framework_score", "performance_band", "avg_sentiment"
]].sort_values("cohort_engagement_percentile", ascending=False).head(10)

display(recent_standouts)

In [ ]:
# Prepare a smaller sample for the scatter plot
scatter_data = video_metrics[["avg_sentiment", "engagement_per_day"]].dropna()
scatter_data = scatter_data.sample(n=min(500, len(scatter_data)), random_state=42)

In [ ]:
# Start the sentiment vs engagement plot
plt.figure(figsize=(10, 6))
sns.regplot(
    data=scatter_data,
    x="avg_sentiment",
    y="engagement_per_day",
    scatter_kws={"alpha": 0.35, "color": colors["orange"]},
    line_kws={"color": colors["blue"], "linewidth": 2},
)

In [ ]:
# Add labels and finish the sentiment plot
plt.title("Sentiment vs Age-Adjusted Engagement")
plt.xlabel("Average sentiment")
plt.ylabel("Engagement per day")
plt.tight_layout()
plt.show()

## Advanced Analysis

Goal:
- Flag videos that may need moderation.
- Score community health by content category.
- Recommend where moderator effort should go.

Note:
Because there are no comment timestamps, this section uses a moderation-risk proxy instead of a true negative sentiment spiral over time.

In [ ]:
# Combine negative sentiment, controversy, scale, and confidence into one moderation score
video_metrics["moderation_priority"] = video_metrics["sample_confidence"] * (
    0.45 * video_metrics["negative_sentiment_percentile"].fillna(0)
    + 0.25 * video_metrics["controversy_percentile"].fillna(0)
    + 0.20 * video_metrics["comment_volume_percentile"].fillna(0)
    + 0.10 * video_metrics["reach_percentile"].fillna(0)
)

In [ ]:
# Turn the moderation score into simple action labels
video_metrics["moderation_flag"] = np.select(
    [
        video_metrics["moderation_priority"] >= 0.85,
        video_metrics["moderation_priority"] >= 0.70,
        video_metrics["moderation_priority"] >= 0.50,
    ],
    ["urgent_review", "watchlist", "monitor"],
    default="healthy",
)

In [ ]:
# Show the videos most likely to need moderation attention
moderation_watchlist = video_metrics[[
    "title", "keywords_joined", "moderation_priority", "moderation_flag",
    "negative_ratio", "controversy_score", "sampled_comments"
]].sort_values("moderation_priority", ascending=False).head(15)

display(moderation_watchlist)

In [ ]:
# Break the video table back out by keyword for category-level health scoring
category_metrics = video_metrics.explode("keywords").rename(columns={"keywords": "keyword"})
category_metrics = category_metrics.dropna(subset=["keyword"])

In [ ]:
# Summarize sentiment and moderation risk by keyword category
category_health = category_metrics.groupby("keyword", as_index=False).agg(
    videos=("video_id", "nunique"),
    avg_sentiment=("avg_sentiment", "mean"),
    avg_negative_ratio=("negative_ratio", "mean"),
    avg_controversy=("controversy_score", "mean"),
    avg_moderation_priority=("moderation_priority", "mean"),
    flagged_video_share=("moderation_priority", lambda s: (s >= 0.70).mean()),
    total_video_comments=("comments", "sum"),
)

In [ ]:
# Convert those category metrics into a community health score
category_health["community_health_score"] = 100 * (
    0.35 * (category_health["avg_sentiment"] / 2)
    + 0.25 * (1 - category_health["avg_negative_ratio"])
    + 0.15 * (1 - category_health["avg_controversy"])
    + 0.15 * (1 - category_health["avg_moderation_priority"])
    + 0.10 * (1 - category_health["flagged_video_share"])
)
category_health = category_health.sort_values("community_health_score")

In [ ]:
# Show the least healthy categories first
display(category_health.head(15))

In [ ]:
# Visualize the categories with the weakest health scores
weakest_categories = category_health.head(12).sort_values("community_health_score")
plt.figure(figsize=(10, 6))
plt.barh(weakest_categories["keyword"], weakest_categories["community_health_score"], color=colors["red"])
plt.title("Lowest Community Health Scores")
plt.xlabel("Community health score")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()

In [ ]:
# Estimate which categories should receive the most moderator time
resource_plan = category_health.copy()
resource_plan["moderator_effort_share_pct"] = resource_plan["avg_moderation_priority"] * np.log1p(resource_plan["total_video_comments"])

In [ ]:
# Convert the raw resource weights into effort percentages
resource_plan["moderator_effort_share_pct"] = 100 * resource_plan["moderator_effort_share_pct"] / resource_plan["moderator_effort_share_pct"].sum()

In [ ]:
# Assign an action level to each category
resource_plan["recommended_action"] = np.select(
    [
        (resource_plan["community_health_score"] < 60) | (resource_plan["moderator_effort_share_pct"] >= 4),
        (resource_plan["community_health_score"] < 72) | (resource_plan["moderator_effort_share_pct"] >= 3),
        (resource_plan["community_health_score"] < 80) | (resource_plan["moderator_effort_share_pct"] >= 2),
    ],
    ["proactive_moderation", "daily_monitoring", "light_monitoring"],
    default="low_touch",
)
resource_plan = resource_plan.sort_values("moderator_effort_share_pct", ascending=False)

In [ ]:
# Show the categories that deserve the most moderation effort
display(resource_plan[[
    "keyword", "community_health_score", "moderator_effort_share_pct",
    "avg_negative_ratio", "avg_controversy", "recommended_action"
]].head(15))

In [ ]:
# Visualize the recommended moderator effort allocation
top_effort = resource_plan.head(12).sort_values("moderator_effort_share_pct")
plt.figure(figsize=(10, 6))
plt.barh(top_effort["keyword"], top_effort["moderator_effort_share_pct"], color=colors["blue"])
plt.title("Recommended Moderator Effort Allocation")
plt.xlabel("Moderator effort share (%)")
plt.ylabel("Keyword")
plt.tight_layout()
plt.show()

## Final Takeaway

This notebook now stays focused on three things:
- simple ranking for beginner analysis,
- a fair time-based performance framework for intermediate analysis,
- moderation and community-health prioritization for advanced analysis.